# Capital Rentals

MATH 153 ~ Group Project ~ Team Members: Karma, Melvin, Risav, Shirjak

In [1]:
import json, re, sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = "data/"

## Part 1. Concatenating datasets

### 1.1 – 1.3 Load, rename, convert, stack

In [2]:
q3 = pd.read_csv(DATA + "listings_q3.csv")
q4 = pd.read_csv(DATA + "listings_q4.csv")

# 1.1 rename Q4 columns to match Q3 exactly (positional order is identical, but we map by name to be explicit)
q4 = q4.rename(columns={
    "ListingID": "listing_id", "Landlord": "landlord_id", "Neighborhood": "nbhd",
    "Bedrooms": "beds", "Bathrooms": "baths", "SqFt": "sqft",
    "Monthly_Rent": "rent", "Date_Listed": "listed_on",
})
assert list(q3.columns) == list(q4.columns)

# 1.2 rent text -> float; dates -> datetime (explicit formats)
q4["rent"] = q4["rent"].str.replace(r"[$,]", "", regex=True).astype(float)
q3["listed_on"] = pd.to_datetime(q3["listed_on"], format="%Y-%m-%d")   # Q3 is ISO
q4["listed_on"] = pd.to_datetime(q4["listed_on"], format="%m/%d/%Y")   # Q4 is MM/DD/YYYY

# 1.3 source column + stack
q3["source"] = "q3"
q4["source"] = "q4"
stacked = pd.concat([q3, q4], ignore_index=True)
print("q3:", q3.shape, "| q4:", q4.shape, "| stacked:", stacked.shape)
stacked.dtypes

q3: (180, 9) | q4: (126, 9) | stacked: (306, 9)


listing_id             object
landlord_id            object
nbhd                   object
beds                    int64
baths                 float64
sqft                  float64
rent                  float64
listed_on      datetime64[ns]
source                 object
dtype: object

**Formats used:** Q3 dates format="%Y-%m-%d", Q4 dates format="%m/%d/%Y". Rent in Q4 was text like `"$4,290"`; we strip `$` and `,` with a regex, then cast to float.

### 1.4 Duplicates (re-posted listings)

In [3]:
dups = stacked[stacked["listing_id"].duplicated(keep=False)].sort_values(["listing_id", "source"])
display(dups)

# To find if the two copies identical apart from 'source'?
cols = [c for c in stacked.columns if c != "source"]
print("Re-posted IDs:", dups["listing_id"].nunique())
print("Copies identical on all other columns:", dups.groupby("listing_id")[cols].nunique().max().max() == 1)

before = len(stacked)
# keep the Q4 copy (the most recent posting). Since copies are identical it makes no numeric difference,
# but the latest posting is the right default if they ever differed.
listings = (stacked.sort_values(["listing_id", "listed_on", "source"])
                   .drop_duplicates("listing_id", keep="last")
                   .sort_values("listing_id").reset_index(drop=True))
print(f"rows before: {before} | rows after: {len(listings)} | removed: {before - len(listings)}")

,listing_id,landlord_id,nbhd,beds,baths,sqft,rent,listed_on,source
36,R-10037,LL123,N11,0,1.0,340.0,1560.0,2026-07-14,q3
303,R-10037,LL123,N11,0,1.0,340.0,1560.0,2026-07-14,q4
126,R-10127,LL121,Nbhd 08,1,1.0,650.0,2740.0,2026-09-18,q3
271,R-10127,LL121,Nbhd 08,1,1.0,650.0,2740.0,2026-09-18,q4
133,R-10134,LL106,n-05,1,1.0,690.0,2430.0,2026-09-18,q3
219,R-10134,LL106,n-05,1,1.0,690.0,2430.0,2026-09-18,q4
138,R-10139,LL114,N04,1,1.0,870.0,3040.0,2026-09-18,q3
251,R-10139,LL114,N04,1,1.0,870.0,3040.0,2026-09-18,q4
143,R-10144,LL109,n10,4,2.0,1720.0,5990.0,2026-09-20,q3
252,R-10144,LL109,n10,4,2.0,1720.0,5990.0,2026-09-20,q4


Re-posted IDs: 6
Copies identical on all other columns: True
rows before: 306 | rows after: 300 | removed: 6


**Finding:** Every re-posted listing is an exact copy (same landlord, rent, date), so which one we keep does not change any number. We keep the later (Q4) copy by convention. Leaving duplicates in would double-count those listings in every average and in the SQL counts.

## Part 2. Regular expressions

### 2.1 Cleaning the neighborhood code

In [4]:
spellings = sorted(listings["nbhd"].unique(), key=lambda s: (s.strip().lower().replace("-", "").replace(" ", "")))
print(len(spellings), "distinct raw spellings:")
print([repr(s) for s in spellings])

59 distinct raw spellings:
["'N01'", "'n01'", "'n-01'", "'N02'", "'n02'", "'n-02'", "'N03'", "'n-03'", "'n03'", "'N04'", "'n-04'", "'n04'", "'n05'", "'N05'", "'n-05'", "'N06'", "'n-06'", "'N07'", "'n07'", "'n-07'", "'n08'", "'N08'", "'n-08'", "'n09'", "'n-09'", "'N09'", "' N1 '", "'n-10'", "' N10 '", "'N10'", "'n10'", "'n-11'", "'N11'", "' N11 '", "'n11'", "'n12'", "'N12'", "'n-12'", "' N12 '", "' N2 '", "' N3 '", "' N4 '", "' N5 '", "' N6 '", "' N7 '", "' N8 '", "' N9 '", "'Nbhd 01'", "'Nbhd 02'", "'Nbhd 03'", "'Nbhd 04'", "'Nbhd 05'", "'Nbhd 06'", "'Nbhd 07'", "'Nbhd 08'", "'Nbhd 09'", "'Nbhd 10'", "'Nbhd 11'", "'Nbhd 12'"]


In [5]:
# One regex: optional spaces, N/n, optional 'bhd', optional space/hyphen, optional leading 0, 1-2 digits, optional spaces.
NBHD_RE = r"^\s*[Nn](?:bhd)?[\s-]*0?(\d{1,2})\s*$"
listings["nbhd_code"] = listings["nbhd"].str.replace(NBHD_RE, lambda m: f"N{int(m.group(1)):02d}", regex=True)
codes = sorted(listings["nbhd_code"].unique())
print(len(codes), codes)
assert len(codes) == 12 and all(re.fullmatch(r"N(0[1-9]|1[0-2])", c) for c in codes)

12 ['N01', 'N02', 'N03', 'N04', 'N05', 'N06', 'N07', 'N08', 'N09', 'N10', 'N11', 'N12']


**Pattern explained:** `^\s*` allows leading spaces; `[Nn]` the letter; `(?:bhd)?` the optional word "bhd" (non-capturing); `[\s-]*` any mix of spaces/hyphens; `0?` an optional leading zero; `(\d{1,2})` captures the number (1 or 2 digits); `\s*$` trailing spaces. The replacement function re-pads the captured number to two digits and puts `N` in front. The assertion proves exactly 12 valid codes (N01–N12) remain.

### 2.2 Parsing raw_ads.txt

In [6]:
PATTERNS = {
    # R- followed by exactly five digits
    "listing_id": r"R-\d{5}",
    # a number (with optional thousands commas) followed by sq ft / sqft / sq.ft. / SF  (case-insensitive)
    "sqft": r"(\d{1,3}(?:,\d{3})+|\d+)\s*(?:sq\.?\s*ft\.?|sqft|SF)\b",
    # dollar sign then digits with optional thousands commas
    "rent": r"\$(\d{1,3}(?:,\d{3})+|\d+)",
    # (ddd) ddd-dddd
    "phone": r"\(\d{3}\)\s*\d{3}-\d{4}",
    # local@domain.tld
    "email": r"[\w.+-]+@[\w-]+(?:\.[\w-]+)*\.[a-z]{2,}",
}

def first(pattern, line, group=0):
    m = re.search(pattern, line, flags=re.IGNORECASE)
    return m.group(group) if m else None

rows = []
with open(DATA + "raw_ads.txt", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        sq, rent = first(PATTERNS["sqft"], line, 1), first(PATTERNS["rent"], line, 1)
        rows.append({
            "listing_id": first(PATTERNS["listing_id"], line),
            "ad_sqft": float(sq.replace(",", "")) if sq else np.nan,
            "ad_rent": float(rent.replace(",", "")) if rent else np.nan,
            "ad_phone": first(PATTERNS["phone"], line),
            "ad_email": first(PATTERNS["email"], line),
        })
ads = pd.DataFrame(rows)
print(ads.shape)
display(ads.head(10))
print(ads.isna().sum())

(32, 5)


,listing_id,ad_sqft,ad_rent,ad_phone,ad_email
0,R-10274,1880.0,6490.0,(301) 555-1370,potomacpropertygroup@rentals.net
1,R-10140,1170.0,2120.0,(202) 555-1185,anacostiahomes@rentals.net
2,R-10155,770.0,2610.0,(240) 555-1407,luis.ortiz@mail.com
3,R-10275,360.0,1520.0,None,hassan.karim@mail.com
4,R-10247,1260.0,5910.0,(301) 555-1555,capitolkeys@rentals.net
5,R-10291,780.0,2930.0,(301) 555-1814,None
6,R-10235,1690.0,6850.0,None,northstarrentals@rentals.net
7,R-10128,1440.0,2840.0,(240) 555-1629,rosa.alvarez@mail.com
8,R-10111,1750.0,3410.0,(301) 555-1222,gloria.diallo@mail.com
9,R-10182,1200.0,2760.0,(301) 555-1333,jamal.wright@mail.com


listing_id    0
ad_sqft       0
ad_rent       0
ad_phone      6
ad_email      9
dtype: int64


**What each pattern does**
- `listing_id` `R-\d{5}`: literal `R-` and exactly five digits.
- `sqft` `(\d{1,3}(?:,\d{3})+|\d+)\s*(?:sq\.?\s*ft\.?|sqft|SF)\b`: captures either a comma-grouped number (`1,260`) or a plain number, then optional space, then any of `sq ft`, `sq.ft.`, `sqft`, `SF`. Case-insensitive; `\b` stops it matching inside longer words.
- `rent` `\$(...)`: the number after a dollar sign, same comma handling. Works for `$6,490 per month`, `$1,520/month`, `$2,930.`
- `phone` `\(\d{3}\)\s*\d{3}-\d{4}`: area code in literal parentheses.
- `email`: word characters/dots before `@`, domain labels, and a final TLD of 2+ letters. Because the TLD must be letters, a sentence-ending period (`...mail.com.`) is not swallowed.

Phone and email are legitimately missing in some ads (some give only one contact method), so those NaNs are expected.

### 2.3 Merge ads onto listings, check rent, fill sqft

In [7]:
merged = listings.merge(ads, on="listing_id", how="outer", indicator=True)
print(merged["_merge"].value_counts())

both = merged[merged["_merge"] == "both"]
print("\nmatched rows:", len(both))
print("rent disagreements (ad_rent vs rent):", (both["ad_rent"] != both["rent"]).sum())
print("ads that had a sqft where listing ALSO had a sqft:", both["sqft"].notna().sum(),
      "| of those, conflicting:", (both["sqft"].notna() & (both["sqft"] != both["ad_sqft"])).sum())

missing_before = listings["sqft"].isna().sum()
listings = listings.merge(ads, on="listing_id", how="left")
listings["sqft"] = listings["sqft"].fillna(listings["ad_sqft"])
missing_after = listings["sqft"].isna().sum()
print(f"\nsqft missing before: {missing_before} | after: {missing_after}")

_merge
left_only     268
both           32
right_only      0
Name: count, dtype: int64

matched rows: 32
rent disagreements (ad_rent vs rent): 0
ads that had a sqft where listing ALSO had a sqft: 0 | of those, conflicting: 0

sqft missing before: 32 | after: 0


**Finding:** The ads agree with the CSV rent on every matched row, so they are a trustworthy second source. They exist exactly for the listings that have no square footage in the CSVs, so `fillna` with `ad_sqft` fills every gap.

Parts 5 and 6: outliers, cleaning, and transformations
The existing Parts 1 and 2 above supply deduplicated listings with square footage recovered from ads. The small setup below joins the registry and neighborhood names to create master for this analysis. It does not replace the separate written answers required for Parts 3 and 4.

All statistical SDs use the sample SD (ddof=1), consistently for detection, masking, and standardization. Outlier flags are review signals, not proof of errors. Raw values are preserved; uncertain corrections are not invented.

In [ ]:
from pathlib import Path
from IPython.display import display

with open(DATA + "landlords.json", encoding="utf-8") as f:
    registry = json.load(f)
landlords = pd.json_normalize(registry["landlords"])
landlords["n_licenses"] = landlords["licenses"].str.len()
neighborhoods = pd.read_csv(DATA + "neighborhoods.csv")
master = (listings.merge(landlords, on="landlord_id", how="left", validate="many_to_one")
          .merge(neighborhoods, on="nbhd_code", how="left", validate="many_to_one"))
assert len(master) == len(listings) == 300
assert master["listing_id"].is_unique
assert master["nbhd_name"].notna().all()
print(f"Master dataset: {master.shape[0]} listings, {master.shape[1]} columns")
uncleaned = master.copy(deep=True)

Part 5. Outliers: z-score, modified z-score, IQR
5.1 Validity first
Check physical/domain constraints before computing statistics. An apartment must have positive rent and floor area; bedrooms must be a nonnegative integer (0 means studio), and bathrooms must be positive. Missing/nonfinite values also require review. A zero floor area is impossible and would produce infinite rent per square foot. A statistical cutoff depends on the distribution and can miss zero or be distorted by extreme values; a domain rule does not.

In [ ]:
numeric = master[["rent", "sqft", "beds", "baths"]]
validity_checks = pd.DataFrame({
    "missing_or_nonfinite": ~np.isfinite(numeric).all(axis=1),
    "nonpositive_rent": master["rent"].le(0),
    "nonpositive_sqft": master["sqft"].le(0),
    "invalid_beds": master["beds"].lt(0) | master["beds"].mod(1).ne(0),
    "nonpositive_baths": master["baths"].le(0),
}, index=master.index)
invalid_mask = validity_checks.any(axis=1)
invalid_rows = master.loc[invalid_mask].copy()
display(invalid_rows[["listing_id", "rent", "sqft", "beds", "baths"]])
display(validity_checks.sum().rename("rows_failing"))
print(f"Invalid rows: {invalid_mask.sum()}; usable for outlier calculations: {(~invalid_mask).sum()}")
# Quarantine invalid rows before division. Keep uncleaned unchanged for 6.3.
analysis = master.loc[~invalid_mask].copy()

Finding: R-10231 has sqft = 0. Its rent is $2,840, so rent alone would not expose the invalid area. No matching ad supplies its true area; exclude this record from the cleaned analysis and preserve it in the audit trail.

5.2 Monthly rent per square foot

In [ ]:
analysis["ppsf"] = analysis["rent"] / analysis["sqft"]
assert np.isfinite(analysis["ppsf"]).all()
display(analysis[["listing_id", "rent", "sqft", "ppsf"]].head())

5.3 Compare three outlier rules
Compute all six sets of flags on the same 299 valid rows, before any statistical exclusions. MAD is the median absolute deviation from the median. Quartiles use pandas' default linear interpolation. Strict inequalities implement the specified thresholds.

In [ ]:
def outlier_rules(series):
    x = series.astype(float)
    if not np.isfinite(x).all():
        raise ValueError("Outlier calculations require finite values.")
    sd = x.std(ddof=1)
    median = x.median()
    mad = (x - median).abs().median()
    if sd == 0 or mad == 0:
        raise ValueError("SD/MAD is zero; the corresponding score is undefined.")
    q1, q3 = x.quantile([0.25, 0.75])
    iqr = q3 - q1
    return pd.DataFrame({
        "z_score": ((x - x.mean()) / sd).abs().gt(3),
        "modified_z": (0.6745 * (x - median) / mad).abs().gt(3.5),
        "IQR": x.lt(q1 - 1.5 * iqr) | x.gt(q3 + 1.5 * iqr),
    }, index=x.index)

flags = pd.DataFrame(index=analysis.index)
comparison_rows = []
for column in ["rent", "ppsf"]:
    rules = outlier_rules(analysis[column])
    for rule in rules:
        flags[f"{column}_{rule}"] = rules[rule]
        ids = analysis.loc[rules[rule], "listing_id"].sort_values().tolist()
        comparison_rows.append({"column": column, "rule": rule,
                                "count": len(ids), "listing_ids": ", ".join(ids)})
comparison = pd.DataFrame(comparison_rows)
display(comparison)
analysis["rent_flag"] = flags.filter(like="rent_").any(axis=1)
analysis["ppsf_flag"] = flags.filter(like="ppsf_").any(axis=1)
display(analysis.loc[flags.any(axis=1),
                     ["listing_id", "nbhd_name", "beds", "baths", "rent", "sqft", "ppsf"]])

5.4 Masking
A very large observation pulls the mean upward and inflates the SD used in the denominator. Other large rents then have smaller absolute z-scores and may be masked. Remove only the largest absolute rent z-score for this experiment; do not silently change the comparison in 5.3.

In [ ]:
rent_z_before = (analysis["rent"] - analysis["rent"].mean()) / analysis["rent"].std(ddof=1)
extreme_index = rent_z_before.abs().idxmax()
extreme_id = analysis.loc[extreme_index, "listing_id"]
without_extreme = analysis.drop(index=extreme_index).copy()
rent_z_after = ((without_extreme["rent"] - without_extreme["rent"].mean())
                / without_extreme["rent"].std(ddof=1))
masking_summary = pd.DataFrame([
    {"sample": "Before", "n": len(analysis), "mean_rent": analysis["rent"].mean(),
     "sample_SD": analysis["rent"].std(ddof=1),
     "flagged_ids": ", ".join(analysis.loc[rent_z_before.abs().gt(3), "listing_id"])},
    {"sample": f"Without {extreme_id}", "n": len(without_extreme),
     "mean_rent": without_extreme["rent"].mean(),
     "sample_SD": without_extreme["rent"].std(ddof=1),
     "flagged_ids": ", ".join(without_extreme.loc[rent_z_after.abs().gt(3), "listing_id"])},
])
display(masking_summary.round(3))
before_ids = set(analysis.loc[rent_z_before.abs().gt(3), "listing_id"])
after_ids = set(without_extreme.loc[rent_z_after.abs().gt(3), "listing_id"])
print("Removed for masking experiment:", extreme_id)
print("Newly exposed IDs:", sorted(after_ids - before_ids))

Finding: Before removal, mean rent is $5,661.80 and sample SD is $38,299.30; only R-10016 ($665,000) exceeds |z| = 3. After removing it, R-10119 and R-10261 exceed the threshold. The robust rules already identify those two on rent because their center and spread are less affected by the single enormous value.

5.5 Flagged on ppsf but not rent
Here “flagged” means flagged by any of the three rules for that column. The per-rule comparison above retains the individual results.

In [ ]:
ppsf_only = analysis.loc[analysis["ppsf_flag"] & ~analysis["rent_flag"],
                          ["listing_id", "beds", "rent", "sqft", "ppsf"]].copy()
diagnoses = {
    "R-10062": "85 sqft for 2 beds suggests an area entry/unit error; $2,970 rent is ordinary.",
    "R-10141": "$27.50 monthly rent suggests a decimal/unit error; area is ordinary; rent lower fences miss it.",
}
ppsf_only["diagnosis"] = ppsf_only["listing_id"].map(diagnoses)
assert ppsf_only["diagnosis"].notna().all()
display(ppsf_only)
rent_q1, rent_q3 = analysis["rent"].quantile([0.25, 0.75])
print("Rent IQR lower fence:", rent_q1 - 1.5 * (rent_q3 - rent_q1))
rent_mad = (analysis["rent"] - analysis["rent"].median()).abs().median()
print("Rent modified-z lower cutoff:", analysis["rent"].median() - 3.5 * rent_mad / 0.6745)

Interpretation: R-10062 has ppsf about $34.94 because its recorded area is only 85 sqft. The numerator ($2,970) looks ordinary, so rent-only checks miss the suspect denominator. R-10141 has ppsf about $0.041: its $27.50 rent is suspiciously low for a 670-sqft one-bedroom. The rent IQR and modified-z lower cutoffs are negative in this broad distribution, and its classical z-score is also small because the extreme high rent inflates the SD. Dividing by area makes the low price stand out more clearly.

These are likely entry errors, but the supplied files do not establish exact replacement values. Derived features reveal inconsistencies that marginal checks miss; inspect both numerator and denominator and seek source confirmation before correcting.

5.6 Decisions and cleaned dataset
The audit below covers the union of all six statistical flags plus the invalid-area row. No matching ads supply verified corrections for the suspect rows. Exclude the three likely entry errors pending verification. Keep the two high-price listings as plausible luxury listings based on size, amenities, neighborhood, and finite ppsf; this is an explicit analytical judgment, not independent verification of the ads.

In [ ]:
decision_notes = {
    "R-10231": ("remove", "Zero sqft is invalid; no source confirms the true area."),
    "R-10016": ("remove", "$665,000/month for 1,420 sqft is a likely entry error; correct rent is unverified."),
    "R-10062": ("remove", "85 sqft for two bedrooms is a likely area/unit error; correct area is unverified."),
    "R-10141": ("remove", "$27.50/month for a 670-sqft one-bedroom is a likely entry error; correct rent is unverified."),
    "R-10119": ("keep", "$14,800 for 2,600 sqft, four beds and 3.5 baths in Georgetown is plausible luxury housing."),
    "R-10261": ("keep", "$9,900 for 1,900 sqft, three beds and 2.5 baths in Navy Yard is high but plausible."),
}
review_ids = set(analysis.loc[flags.any(axis=1), "listing_id"]) | set(invalid_rows["listing_id"])
assert review_ids == set(decision_notes), "Every flagged/invalid row must have a decision."
audit = master.loc[master["listing_id"].isin(review_ids),
                   ["listing_id", "nbhd_name", "beds", "baths", "rent", "sqft"]].copy()
audit["decision"] = audit["listing_id"].map(lambda k: decision_notes[k][0])
audit["reason"] = audit["listing_id"].map(lambda k: decision_notes[k][1])
display(audit.sort_values("listing_id"))
remove_ids = set(audit.loc[audit["decision"].eq("remove"), "listing_id"])
cleaned = master.loc[~master["listing_id"].isin(remove_ids)].copy()
cleaned["ppsf"] = cleaned["rent"] / cleaned["sqft"]
assert len(cleaned) == 296
assert cleaned[["rent", "sqft"]].gt(0).all().all()
assert np.isfinite(cleaned[["rent", "sqft", "ppsf"]]).all().all()
print(f"Original unique listings: {len(master)}; removed: {len(remove_ids)}; retained: {len(cleaned)}")

Part 6. Scaling and transformations
6.1 Transform rent and sqft and verify
Keep original units alongside the new columns. Each transformation is fitted to the cleaned dataset. Standardization uses sample SD (ddof=1) so the verification uses the same convention. These are descriptive transformations; for predictive modeling fit parameters on training data only.

In [ ]:
transformed = cleaned.copy()
verification_rows = []
for column in ["rent", "sqft"]:
    x = cleaned[column]
    span = x.max() - x.min()
    sd = x.std(ddof=1)
    assert span > 0 and sd > 0
    transformed[f"{column}_minmax"] = (x - x.min()) / span
    transformed[f"{column}_standard"] = (x - x.mean()) / sd
    transformed[f"{column}_log1p"] = np.log1p(x)
    mm = transformed[f"{column}_minmax"]
    standardized = transformed[f"{column}_standard"]
    assert mm.between(0, 1).all()
    assert np.isclose(mm.min(), 0) and np.isclose(mm.max(), 1)
    assert np.isclose(standardized.mean(), 0, atol=1e-12)
    assert np.isclose(standardized.std(ddof=1), 1)
    verification_rows.append({"column": column, "minmax_min": mm.min(),
                              "minmax_max": mm.max(), "standard_mean": standardized.mean(),
                              "standard_sample_SD": standardized.std(ddof=1)})
verification = pd.DataFrame(verification_rows)
display(verification)
display(transformed[["listing_id", "rent", "rent_minmax", "rent_standard", "rent_log1p",
                     "sqft", "sqft_minmax", "sqft_standard", "sqft_log1p"]].head())

6.2 Skewness and side-by-side histograms

In [ ]:
skewness = pd.Series({"rent": cleaned["rent"].skew(),
                      "log1p_rent": transformed["rent_log1p"].skew()}, name="sample_skewness")
display(skewness)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
axes[0].hist(cleaned["rent"], bins=25, color="#246a8a", edgecolor="white")
axes[0].set(title=f"Cleaned rent (skew = {skewness['rent']:.3f})",
            xlabel="Monthly rent ($)", ylabel="Number of listings")
axes[1].hist(transformed["rent_log1p"], bins=25, color="#d68b36", edgecolor="white")
axes[1].set(title=f"log1p(rent) (skew = {skewness['log1p_rent']:.3f})",
            xlabel="log(1 + monthly rent)", ylabel="Number of listings")
plt.show()
print(f"Skewness changes from {skewness['rent']:.4f} to {skewness['log1p_rent']:.4f}.")

Choice: Use log1p for this right-skewed price distribution when reducing the influence of its long upper tail is useful. It compresses large prices more than small ones and reduces skewness here. Min-max scaling and standardization are linear changes of units, so neither removes skewness. Log transformation need not produce a perfectly symmetric or normal distribution.

6.3 Scaling before versus after cleaning
Fit min-max scaling once on all 300 deduplicated, uncleaned rents (including the extremes), and once on the cleaned rents. Compare the same retained listings under both sets of scaling parameters so that row membership cannot explain the difference.

In [ ]:
raw_rent = uncleaned["rent"]
raw_min, raw_max = raw_rent.min(), raw_rent.max()
uncleaned["rent_minmax_uncleaned"] = (raw_rent - raw_min) / (raw_max - raw_min)
scale_comparison = cleaned[["listing_id", "rent"]].copy()
scale_comparison["scaled_using_uncleaned"] = (cleaned["rent"] - raw_min) / (raw_max - raw_min)
scale_comparison["scaled_using_cleaned"] = transformed["rent_minmax"]
display(scale_comparison[["scaled_using_uncleaned", "scaled_using_cleaned"]]
        .quantile([0, 0.25, 0.5, 0.75, 0.95, 1]).rename_axis("quantile"))
display(scale_comparison.sort_values("rent").iloc[[0, len(scale_comparison)//2, -1]])
print(f"Uncleaned rent range: ${raw_min:,.2f} to ${raw_max:,.2f}")
print(f"Cleaned rent range: ${cleaned['rent'].min():,.2f} to ${cleaned['rent'].max():,.2f}")
print(f"Even the largest retained rent maps to only {scale_comparison['scaled_using_uncleaned'].max():.4f} using the uncleaned range.")

Finding: The $665,000 maximum stretches the uncleaned range so much that ordinary listings cluster close to zero and useful differences are compressed. Even the highest retained rent maps to about 0.0222. Clean first, then estimate scaling parameters. Keeping plausible luxury listings still affects min-max scaling, but silently deleting real high prices just to improve a scale would distort the market.

6.4 Why log(1 + x)?
For nonnegative data, log1p(0) = 0, while log(0) is undefined (negative infinity in NumPy). In this project beds includes zeros for studios, so np.log1p(cleaned["beds"]) would retain them as finite values. This is an example of where the distinction matters, not a claim that bedroom counts need a log transform. NumPy's log1p is also more numerically accurate for tiny x. It is undefined for real-valued inputs below -1 and does not make negative or invalid measurements acceptable.

These files are generated inside results/ when the notebook is run. cleaned and transformed remain available for later project sections. The exports preserve original measurement values; the audit records every exclusion and reviewed retention.

In [ ]:
results_dir = Path("results")
results_dir.mkdir(exist_ok=True)
cleaned.to_csv(results_dir / "listings_cleaned.csv", index=False)
transformed.to_csv(results_dir / "listings_transformed.csv", index=False)
audit.to_csv(results_dir / "cleaning_audit.csv", index=False)
comparison.to_csv(results_dir / "outlier_comparison.csv", index=False)
fig.savefig(results_dir / "rent_histograms.png", dpi=160, bbox_inches="tight")
print("Saved cleaned data, transformations, decisions, outlier comparison, and histogram figure to results/.")